# FastText for Language Identification

**Language identification** (LangID) is the task of determining what language a piece of text is written in.  
It is a foundational step in any multilingual NLP pipeline.

## Why FastText?

FastText (Joulin et al., 2016) extends Word2Vec with **subword n-grams**: instead of treating words as atomic units, it breaks them into character n-grams.

| Model | Unit | Handles OOV? | Good for morphology? |
|---|---|---|---|
| Word2Vec | whole words | No | No |
| **FastText** | word + character n-grams | **Yes** | **Yes** |

For language ID this is crucial: languages differ dramatically at the **character level** (scripts, diacritics, letter combinations), so subword features are highly discriminative.

```
"playing"  →  play, playing, #pl, #pla, #play, lay, layi, ..., ing#
                               ↑ character n-grams fed into the model
```

## This notebook covers

1. Using Meta's **pre-trained** `lid.176` model (176 languages, ~900 KB)
2. **Training your own** language ID model from scratch on the `papluca/language-identification` dataset (20 languages)
3. **Evaluating** both models with accuracy, F1, and confusion matrix
4. Testing **edge cases**: short texts, code-switching, transliterated text

In [ ]:
# fasttext-wheel is a pre-compiled wheel that avoids the C++ build step
!pip install -q fasttext-wheel datasets scikit-learn matplotlib seaborn

In [ ]:
import os
import re
import time
import urllib.request
import tempfile
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import fasttext
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

plt.style.use('ggplot')
%matplotlib inline

os.makedirs('models', exist_ok=True)
os.makedirs('data',   exist_ok=True)

print(f"fasttext version: {fasttext.__version__}")

---
## 1. Pre-trained Language Identification Model

Meta released two pre-trained LangID models:

| Model | Size | Languages | Notes |
|---|---|---|---|
| `lid.176.bin` | 126 MB | 176 | Full accuracy |
| `lid.176.ftz` | 917 KB | 176 | Compressed (~1% accuracy drop) |

We'll use the compressed `.ftz` model — it fits in a few seconds and is good enough for demos.

In [ ]:
MODEL_PATH = 'models/lid.176.ftz'
MODEL_URL  = 'https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz'

if not os.path.exists(MODEL_PATH):
    print("Downloading lid.176.ftz (~900 KB)...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Done.")
else:
    print("Model already downloaded.")

# Load — suppress the 'Warning: loading model with args' message
pretrained = fasttext.load_model(MODEL_PATH)
print(f"Model loaded. Supports {len(pretrained.labels)} languages.")

In [ ]:
def predict_lang(model, text, k=1):
    """Return top-k (language, confidence) predictions. Cleans newlines first."""
    text = text.replace('\n', ' ').strip()
    labels, probs = model.predict(text, k=k)
    # Labels look like '__label__en' — strip the prefix
    langs = [l.replace('__label__', '') for l in labels]
    return list(zip(langs, probs))

# Quick sanity check across 8 languages
test_texts = [
    ("en", "The quick brown fox jumps over the lazy dog."),
    ("fr", "Le renard brun rapide saute par-dessus le chien paresseux."),
    ("de", "Der schnelle braune Fuchs springt über den faulen Hund."),
    ("es", "El veloz zorro marrón salta sobre el perro perezoso."),
    ("it", "La volpe marrone veloce salta sopra il cane pigro."),
    ("ar", "الثعلب البني السريع يقفز فوق الكلب الكسول."),
    ("zh", "快速的棕色狐狸跳过了懒狗。"),
    ("he", "השועל החום המהיר קופץ מעל הכלב העצלן."),
]

print(f"{'True':5s}  {'Predicted':8s}  {'Conf':6s}  Text (truncated)")
print("-" * 65)
for true_lang, text in test_texts:
    pred, conf = predict_lang(pretrained, text)[0]
    mark = "✓" if pred == true_lang else "✗"
    print(f"{true_lang:5s}  {pred:8s}  {conf:.4f}  {mark}  {text[:40]}")

In [ ]:
# Top-5 predictions show the model's uncertainty
ambiguous_texts = [
    "OK",                      # too short — ambiguous
    "I love you",              # short English
    "resume",                  # English word, also looks French
    "Un kilo de tomates s'il vous plaît.",  # clear French
]

for text in ambiguous_texts:
    top5 = predict_lang(pretrained, text, k=5)
    print(f"\nText: '{text}'")
    for lang, conf in top5:
        bar = '█' * int(conf * 30)
        print(f"  {lang:4s} {conf:.3f}  {bar}")

In [ ]:
# What languages does the model support?
all_langs = sorted([l.replace('__label__', '') for l in pretrained.labels])

# Map ISO 639-1 / 639-3 codes to readable names for the ones we'll use
LANG_NAMES = {
    'en': 'English',   'fr': 'French',    'de': 'German',
    'es': 'Spanish',   'it': 'Italian',   'pt': 'Portuguese',
    'nl': 'Dutch',     'pl': 'Polish',    'ru': 'Russian',
    'sv': 'Swedish',   'da': 'Danish',    'fi': 'Finnish',
    'tr': 'Turkish',   'ar': 'Arabic',    'zh': 'Chinese',
    'ja': 'Japanese',  'ko': 'Korean',    'hi': 'Hindi',
    'he': 'Hebrew',    'ro': 'Romanian',
}

print(f"Total languages supported: {len(all_langs)}")
print("\nSample (first 40):")
for i, lang in enumerate(all_langs[:40]):
    name = LANG_NAMES.get(lang, '')
    print(f"  {lang:8s} {name}", end='\n' if (i+1) % 4 == 0 else '')

---
## 2. Edge Cases

Real-world text is messy. Let's probe where the pre-trained model struggles.

In [ ]:
edge_cases = [
    # (description, text)
    ("Very short (1 word)",         "Hello"),
    ("Very short (2 words)",        "Hello world"),
    ("Emoji only",                  "😂😂😂"),
    ("Numbers only",                "12345 67890"),
    ("URL",                         "https://www.example.com/page?q=test"),
    ("Code snippet",                "def foo(x): return x + 1"),
    ("Code-switching EN/FR",        "I went to the marché this morning"),
    ("Code-switching EN/ES",        "I need to hablar with you"),
    ("Transliterated Arabic",       "ana behibak kteer"),      # 'I love you a lot' in Arabic transliteration
    ("Transliterated Hebrew",       "ma nishma? hakol beseder"),  # 'What's up? All good'
    ("Mixed script",                "This is English and هذا عربي"),
    ("All caps",                    "THIS IS SHOUTING IN ENGLISH"),
    ("Typos",                       "Ths sentnce has tyops"),
]

print(f"{'Description':30s}  {'Top prediction':10s}  {'Confidence':10s}  2nd guess")
print("-" * 75)
for desc, text in edge_cases:
    top2 = predict_lang(pretrained, text, k=2)
    l1, c1 = top2[0]
    l2, c2 = top2[1] if len(top2) > 1 else ('?', 0)
    print(f"{desc:30s}  {l1:10s}  {c1:.4f}      {l2} ({c2:.3f})")

In [ ]:
# How does text length affect confidence?
base_sentence = "The history of artificial intelligence dates back to the early days of computing."
words = base_sentence.split()

lengths, confidences = [], []
for n in range(1, len(words) + 1):
    snippet = ' '.join(words[:n])
    pred, conf = predict_lang(pretrained, snippet)[0]
    lengths.append(n)
    confidences.append(conf)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lengths, confidences, marker='o', color='#3498db')
ax.axhline(y=0.9, color='gray', linestyle='--', label='90% confidence')
ax.set_xlabel('Number of words')
ax.set_ylabel('Model confidence')
ax.set_title('Language ID Confidence vs Text Length')
ax.legend()
for n, c in zip(lengths, confidences):
    if n <= 5 or n == len(words):
        ax.annotate(f"{c:.2f}", (n, c), textcoords='offset points', xytext=(0, 8), fontsize=8)
plt.tight_layout()
plt.show()
print("Confidence rises quickly and plateaus — even a few words are usually enough.")

---
## 3. Collecting Language Data from Wikipedia

Wikipedia is one of the largest multilingual text corpora in existence.  
The **Wikimedia sitematrix API** tells us every Wikipedia language edition that exists — over 1,000 languages.

We can use this to:
1. Discover all available language codes and their native names
2. Fetch real Wikipedia article text per language as training data

This gives us a **freely available, labeled, high-quality multilingual corpus** — no downloads required.

In [ ]:
import requests
import pandas as pd

# Fetch the full list of Wikipedia language editions from the Wikimedia sitematrix API
params = {'action': 'sitematrix', 'smtype': 'language', 'format': 'json'}
languages = requests.get('https://commons.wikimedia.org/w/api.php?', params=params).json()

print(f"Found {languages['sitematrix']['count']} languages")

In [ ]:
# Parse each language entry into a structured list — exactly as shown in the lecture
language_names = []
for language_index in range(languages['sitematrix']['count']):
    if str(language_index) in languages['sitematrix']:
        current_language = languages['sitematrix'][str(language_index)]
        language_names.append({
            'code':      current_language['code'],
            'name':      current_language['name'],
            'localname': current_language['localname'] if ('localname' in current_language) else ''
        })

language_names = pd.DataFrame(language_names)

print(f"Parsed {len(language_names)} language entries")
print(f"\nSample rows:")
display(language_names.head(15))

In [ ]:
# Explore the language landscape
has_localname = language_names['localname'].str.strip().str.len() > 0
print(f"Total language editions:               {len(language_names)}")
print(f"With native-script name (localname):   {has_localname.sum()}")

# Languages where English name ≠ native name (different scripts or spellings)
different = language_names[
    (language_names['name'] != language_names['localname']) & has_localname
]
print(f"English name ≠ native name:            {len(different)}")
print("\nSample — English name vs native name:")
display(different[['code', 'name', 'localname']].head(20))

# Which of these languages does the pre-trained FastText model support?
ft_lang_codes = set(l.replace('__label__', '') for l in pretrained.labels)
wiki_codes = set(language_names['code'])
overlap = ft_lang_codes & wiki_codes
print(f"\nPre-trained model covers {len(overlap)} / {len(wiki_codes)} Wikipedia language editions")

In [ ]:
def fetch_wikipedia_extract(lang_code, n_sentences=5, timeout=5):
    """
    Fetch a random Wikipedia article intro in the given language.
    Returns (title, text) or None on failure.
    """
    url = f'https://{lang_code}.wikipedia.org/w/api.php'
    params = {
        'action':       'query',
        'generator':    'random',
        'grnnamespace': 0,          # article namespace only
        'prop':         'extracts',
        'exsentences':  n_sentences,
        'exintro':      True,
        'explaintext':  True,       # plain text, no HTML
        'format':       'json',
    }
    try:
        r = requests.get(url, params=params, timeout=timeout)
        pages = r.json().get('query', {}).get('pages', {})
        if not pages:
            return None
        page = next(iter(pages.values()))
        extract = page.get('extract', '').strip()
        title   = page.get('title', '')
        if extract and len(extract) > 40:
            return title, extract
    except Exception:
        pass
    return None


# Fetch one article per language and test the pre-trained model on it
FETCH_LANGS = ['en', 'fr', 'de', 'es', 'it', 'pt', 'nl', 'pl',
               'ru', 'ar', 'zh', 'ja', 'ko', 'hi', 'he', 'tr',
               'sv', 'fi', 'da', 'ro']

print(f"Fetching one random Wikipedia article per language ({len(FETCH_LANGS)} langs)...")
wiki_samples = {}
for lang in FETCH_LANGS:
    result = fetch_wikipedia_extract(lang, n_sentences=3)
    if result:
        wiki_samples[lang] = result
    else:
        print(f"  [{lang}] FAILED")

print(f"Fetched {len(wiki_samples)} / {len(FETCH_LANGS)} articles\n")

print(f"{'Lang':5s}  {'Predicted':9s}  {'Conf':6s}  Title / extract")
print("-" * 80)
for lang, (title, extract) in wiki_samples.items():
    pred, conf = predict_lang(pretrained, extract)[0]
    mark = '✓' if pred == lang else '✗'
    print(f"{lang:5s}  {pred:9s}  {conf:.3f}  {mark}  [{title[:22]}] {extract[:40]}...")

In [ ]:
# Build a training corpus from Wikipedia — fetch multiple articles per language
WIKI_TRAIN_FILE  = 'data/langid_wiki_train.txt'
ARTICLES_PER_LANG = 30   # fetch 30 articles per language

print(f"Fetching {ARTICLES_PER_LANG} articles × {len(FETCH_LANGS)} languages...")
print("(Makes real HTTP requests — may take ~1–2 minutes)\n")

total_written = 0
with open(WIKI_TRAIN_FILE, 'w', encoding='utf-8') as f:
    for lang in FETCH_LANGS:
        lang_count = 0
        for _ in range(ARTICLES_PER_LANG * 3):   # over-fetch to cover failures
            if lang_count >= ARTICLES_PER_LANG:
                break
            result = fetch_wikipedia_extract(lang, n_sentences=5)
            if result:
                _, text = result
                # Split into sentences → more, shorter training examples
                for sent in text.split('.'):
                    sent = sent.replace('\n', ' ').strip()
                    if len(sent) > 30:
                        f.write(f"__label__{lang} {sent}\n")
                        total_written += 1
                lang_count += 1
        print(f"  [{lang}] {lang_count} articles written")

print(f"\nTotal training lines: {total_written:,} → {WIKI_TRAIN_FILE}")

In [ ]:
# Train a FastText model on Wikipedia data and compare with the papluca-trained model
print("Training FastText on Wikipedia corpus...")
start = time.time()

model_wiki = fasttext.train_supervised(
    input=WIKI_TRAIN_FILE,
    epoch=30,
    lr=1.0,
    wordNgrams=2,
    minn=2,
    maxn=4,
    dim=64,
    loss='ova',
    thread=4,
)
print(f"Done in {time.time()-start:.1f}s")

# Evaluate on the papluca test set (held-out, not used in training)
y_pred_wiki = batch_predict(model_wiki, texts)

# Restrict to languages the Wikipedia model was trained on
wiki_mask = [i for i, lang in enumerate(y_true) if lang in FETCH_LANGS]
y_true_m  = [y_true[i]          for i in wiki_mask]
y_wiki_m  = [y_pred_wiki[i]     for i in wiki_mask]
y_tuned_m = [y_pred_tuned[i]    for i in wiki_mask]
y_pre_m   = [y_pred_pretrained[i] for i in wiki_mask]

print(f"\nTest accuracy on {len(FETCH_LANGS)} shared languages (papluca test set):")
print(f"  Wikipedia-trained model:   {accuracy_score(y_true_m, y_wiki_m):.4f}")
print(f"  papluca-trained (tuned):   {accuracy_score(y_true_m, y_tuned_m):.4f}")
print(f"  Pre-trained lid.176.ftz:   {accuracy_score(y_true_m, y_pre_m):.4f}")
print()
print("Key insight: even with only ~600 Wikipedia sentences per language,")
print("FastText achieves competitive accuracy thanks to character n-gram features.")

---
## 4. Training a Custom Language ID Model (Curated Dataset)

FastText supervised training needs data in this format (one line per example):

```
__label__en  this is an english sentence
__label__fr  voici une phrase en français
__label__de  das ist ein deutscher satz
```

We'll use the **`papluca/language-identification`** dataset from Hugging Face — 90 K examples across 20 languages.

In [ ]:
from datasets import load_dataset

print("Loading papluca/language-identification...")
dataset = load_dataset('papluca/language-identification')

train_ds = dataset['train']
val_ds   = dataset['validation']
test_ds  = dataset['test']

print(f"Train:      {len(train_ds):,} examples")
print(f"Validation: {len(val_ds):,} examples")
print(f"Test:       {len(test_ds):,} examples")
print(f"\nSample entry: {train_ds[0]}")

# Count per language
lang_counts = Counter(train_ds['labels'])
print(f"\nLanguages ({len(lang_counts)}): {sorted(lang_counts.keys())}")

In [ ]:
# Dataset distribution
langs_sorted = sorted(lang_counts.keys())
counts = [lang_counts[l] for l in langs_sorted]
labels = [LANG_NAMES.get(l, l) for l in langs_sorted]

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(labels, counts, color='#3498db', alpha=0.8)
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_ylabel('Training examples')
ax.set_title('Examples per Language in Training Set')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            str(count), ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

print("\nSample texts per language:")
shown = set()
for ex in train_ds:
    lang = ex['labels']
    if lang not in shown:
        name = LANG_NAMES.get(lang, lang)
        print(f"  [{lang}] {name:12s}: {ex['text'][:70]}")
        shown.add(lang)
    if len(shown) == len(lang_counts):
        break

In [ ]:
def write_fasttext_file(split, path):
    """Write a dataset split to FastText training format."""
    with open(path, 'w', encoding='utf-8') as f:
        for ex in split:
            # Collapse newlines; FastText expects one example per line
            text = ex['text'].replace('\n', ' ').strip()
            lang = ex['labels']
            f.write(f"__label__{lang} {text}\n")
    print(f"Wrote {len(split):,} lines to {path}")

TRAIN_FILE = 'data/langid_train.txt'
VAL_FILE   = 'data/langid_val.txt'
TEST_FILE  = 'data/langid_test.txt'

write_fasttext_file(train_ds, TRAIN_FILE)
write_fasttext_file(val_ds,   VAL_FILE)
write_fasttext_file(test_ds,  TEST_FILE)

# Peek at the first few lines
print("\nFirst 3 lines of training file:")
with open(TRAIN_FILE, encoding='utf-8') as f:
    for i, line in enumerate(f):
        print(f"  {line[:80].rstrip()}")
        if i == 2:
            break

### 4.1 Train with Default Hyperparameters

In [ ]:
print("Training FastText language ID model (default params)...")
start = time.time()

model_default = fasttext.train_supervised(
    input=TRAIN_FILE,
    epoch=5,
    lr=0.1,
    wordNgrams=2,     # use word unigrams + bigrams
    dim=16,           # embedding dimension
    loss='softmax',
)

elapsed = time.time() - start
print(f"Training took {elapsed:.1f}s")

# Quick eval on validation set
n_val, precision, recall = model_default.test(VAL_FILE)
print(f"\nValidation set (n={n_val:,})")
print(f"  Precision @ 1: {precision:.4f}")
print(f"  Recall    @ 1: {recall:.4f}")

### 4.2 Train with Better Hyperparameters

FastText's **character n-grams** (`minn`/`maxn`) are the key to language ID.  
Languages differ at the character level (scripts, diacritics), so character-level features are highly discriminative.

| Parameter | Meaning |
|---|---|
| `minn` / `maxn` | min/max length of character n-grams |
| `wordNgrams` | also use word n-gram features |
| `dim` | embedding dimension |
| `lr` | learning rate |
| `epoch` | training passes |

In [ ]:
print("Training with character n-grams (better for language ID)...")
start = time.time()

model_tuned = fasttext.train_supervised(
    input=TRAIN_FILE,
    epoch=25,
    lr=1.0,
    wordNgrams=2,
    minn=2,          # character n-grams of length 2 to 4
    maxn=4,
    dim=64,
    loss='ova',      # one-vs-all: better for many-class problems
    thread=4,
)

elapsed = time.time() - start
print(f"Training took {elapsed:.1f}s")

n_val, precision, recall = model_tuned.test(VAL_FILE)
print(f"\nValidation set (n={n_val:,})")
print(f"  Precision @ 1: {precision:.4f}")
print(f"  Recall    @ 1: {recall:.4f}")

In [ ]:
model_tuned.save_model('models/langid_custom.bin')
size_mb = os.path.getsize('models/langid_custom.bin') / 1e6
print(f"Custom model saved ({size_mb:.1f} MB)")

# Compress with quantization
model_tuned.quantize(input=TRAIN_FILE, retrain=True, qnorm=True, cutoff=50000)
model_tuned.save_model('models/langid_custom.ftz')
size_ftz = os.path.getsize('models/langid_custom.ftz') / 1e6
print(f"Compressed model saved ({size_ftz:.1f} MB)")
print(f"Compression ratio: {size_mb/size_ftz:.1f}x")

---
## 5. Evaluation

Let's evaluate on the held-out test set and compare all models.

In [ ]:
# Collect test predictions for all three models
y_true = [ex['labels'] for ex in test_ds]
texts  = [ex['text'].replace('\n', ' ').strip() for ex in test_ds]

def batch_predict(model, texts):
    """Return predicted language code for each text."""
    labels, _ = model.predict(texts)
    return [l[0].replace('__label__', '') for l in labels]

print("Predicting on test set...")
y_pred_pretrained = batch_predict(pretrained,    texts)
y_pred_default    = batch_predict(model_default, texts)
y_pred_tuned      = batch_predict(model_tuned,   texts)

acc_pretrained = accuracy_score(y_true, y_pred_pretrained)
acc_default    = accuracy_score(y_true, y_pred_default)
acc_tuned      = accuracy_score(y_true, y_pred_tuned)

print(f"\n{'Model':30s}  {'Test Accuracy':>13s}")
print("-" * 46)
print(f"{'Pre-trained lid.176.ftz':30s}  {acc_pretrained:.4f}")
print(f"{'Custom (default params)':30s}  {acc_default:.4f}")
print(f"{'Custom (tuned + char ngrams)':30s}  {acc_tuned:.4f}")

In [ ]:
# Per-language accuracy for the tuned model
lang_correct = defaultdict(int)
lang_total   = defaultdict(int)

for true, pred in zip(y_true, y_pred_tuned):
    lang_total[true] += 1
    if true == pred:
        lang_correct[true] += 1

per_lang = sorted(
    [(lang, lang_correct[lang] / lang_total[lang]) for lang in lang_total],
    key=lambda x: x[1]
)

langs_plot = [LANG_NAMES.get(l, l) for l, _ in per_lang]
accs_plot  = [a for _, a in per_lang]

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if a < 0.95 else '#2ecc71' for a in accs_plot]
bars = ax.barh(langs_plot, accs_plot, color=colors, alpha=0.85)
ax.axvline(x=1.0, color='gray', linestyle='--', linewidth=1)
ax.set_xlim(0.8, 1.02)
ax.set_xlabel('Accuracy')
ax.set_title('Per-Language Test Accuracy (Custom Tuned Model)')
for bar, acc in zip(bars, accs_plot):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f"{acc:.3f}", va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix — only for the 20 languages in our custom model
custom_langs = sorted(set(y_true))
display_labels = [LANG_NAMES.get(l, l) for l in custom_langs]

cm = confusion_matrix(y_true, y_pred_tuned, labels=custom_langs)
# Normalize by row (true label) so each row sums to 1
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax.set_xticks(range(len(custom_langs)))
ax.set_yticks(range(len(custom_langs)))
ax.set_xticklabels(display_labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(display_labels, fontsize=9)
ax.set_xlabel('Predicted language')
ax.set_ylabel('True language')
ax.set_title('Confusion Matrix — Custom Tuned Model (row-normalized)', fontsize=13)

# Annotate cells with errors only (off-diagonal)
for i in range(len(custom_langs)):
    for j in range(len(custom_langs)):
        val = cm_norm[i, j]
        if val > 0.01:   # only annotate cells with >1% probability
            color = 'white' if val > 0.5 else 'black'
            ax.text(j, i, f"{val:.2f}", ha='center', va='center',
                    fontsize=7, color=color)

plt.tight_layout()
plt.show()
print("Off-diagonal entries show which language pairs are most confused.")

In [ ]:
# Analyse the most common mistakes
mistakes = [(true, pred, text) for true, pred, text in zip(y_true, y_pred_tuned, texts)
            if true != pred]

mistake_pairs = Counter((true, pred) for true, pred, _ in mistakes)
print(f"Total errors: {len(mistakes)} / {len(y_true)} ({100*len(mistakes)/len(y_true):.1f}%)")
print(f"\nTop 10 confusion pairs (true → predicted):")
for (true, pred), count in mistake_pairs.most_common(10):
    t_name = LANG_NAMES.get(true, true)
    p_name = LANG_NAMES.get(pred, pred)
    print(f"  {t_name:12s} → {p_name:12s}: {count} errors")

# Show example mistakes for the most confusable pair
top_pair = mistake_pairs.most_common(1)[0][0]
examples = [(t, p, txt) for t, p, txt in mistakes if (t, p) == top_pair][:3]
print(f"\nExample '{top_pair[0]}' → '{top_pair[1]}' errors:")
for _, _, txt in examples:
    print(f"  {txt[:90]}")

---
## 6. Model Comparison: Pre-trained vs Custom vs Wikipedia

| Model | Training data | Languages | Notes |
|---|---|---|---|
| `lid.176.ftz` | Wikipedia + OpenSubtitles | 176 | Ready to use, tiny size |
| Custom (papluca) | Curated HF dataset | 20 | Balanced, domain-matched |
| Custom (Wikipedia API) | Live Wikipedia articles | 20 | Built from scratch, no downloads |

In [ ]:
# Side-by-side accuracy per language for all three models
acc_by_lang = {}
model_runs = [
    ('Pre-trained lid.176', y_pred_pretrained),
    ('Custom (papluca)',    y_pred_tuned),
    ('Wikipedia API',       y_pred_wiki),
]
for model_name, preds in model_runs:
    correct = defaultdict(int)
    total   = defaultdict(int)
    for true, pred in zip(y_true, preds):
        total[true] += 1
        if true == pred:
            correct[true] += 1
    acc_by_lang[model_name] = {lang: correct[lang]/total[lang] for lang in total}

common_langs = sorted(acc_by_lang['Pre-trained lid.176'].keys())
x = np.arange(len(common_langs))
width = 0.26
names = [LANG_NAMES.get(l, l) for l in common_langs]

fig, ax = plt.subplots(figsize=(15, 5))
colors = ['#3498db', '#e74c3c', '#2ecc71']
for i, (model_name, _) in enumerate(model_runs):
    vals = [acc_by_lang[model_name].get(l, 0) for l in common_langs]
    ax.bar(x + (i - 1) * width, vals, width, label=model_name, color=colors[i], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_ylim(0.6, 1.05)
ax.set_ylabel('Accuracy')
ax.set_title('Per-Language Accuracy: All Models Compared')
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Practical Usage Patterns

In production, you often want to:
- **Filter by confidence** (reject low-confidence predictions)
- **Detect code-switching** (multiple languages in one document)
- **Process at scale** (batch prediction)

In [ ]:
def identify_language(model, text, threshold=0.5):
    """
    Identify the language of text.
    Returns (language, confidence) or ('unknown', 0) if below threshold.
    """
    if not text or not text.strip():
        return 'unknown', 0.0
    lang, conf = predict_lang(model, text)[0]
    if conf < threshold:
        return 'unknown', conf
    return lang, conf


def detect_code_switching(model, text, sentence_min_len=15):
    """
    Split text into sentences and check if multiple languages appear.
    Returns a list of (sentence, language, confidence) tuples.
    """
    # Simple sentence split on ., !, ?
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    results = []
    for sent in sentences:
        if len(sent) >= sentence_min_len:
            lang, conf = identify_language(model, sent, threshold=0.3)
            results.append((sent, lang, conf))
    return results


# --- Demo: confidence filtering ---
texts_with_expected = [
    ("This is a clear English sentence.",    "en"),
    ("Cette phrase est clairement en français.", "fr"),
    ("OK",                                    "???"),   # too short
    ("lol xD",                               "???"),   # too short
]

print("Language detection with confidence threshold=0.6:")
print(f"  {'Text':45s}  {'Lang':8s}  {'Conf':6s}")
print("  " + "-" * 65)
for text, expected in texts_with_expected:
    lang, conf = identify_language(pretrained, text, threshold=0.6)
    print(f"  {text:45s}  {lang:8s}  {conf:.3f}")

In [ ]:
# Detect code-switching in a multilingual paragraph
multilingual_text = """
Artificial intelligence is transforming our world.
L'intelligence artificielle transforme notre monde.
Die künstliche Intelligenz verändert unsere Welt.
La inteligencia artificial está transformando nuestro mundo.
الذكاء الاصطناعي يحول عالمنا.
""".strip()

results = detect_code_switching(pretrained, multilingual_text)
unique_langs = set(r[1] for r in results if r[1] != 'unknown')

print(f"Detected {len(unique_langs)} languages in the text:\n")
for sent, lang, conf in results:
    name = LANG_NAMES.get(lang, lang)
    print(f"  [{lang}] {name:12s} ({conf:.2f})  →  {sent[:60]}")

In [ ]:
# Benchmark batch prediction speed
sample_texts = texts[:1000]

# FastText batch predict (all at once)
start = time.time()
_ = pretrained.predict(sample_texts)
batch_time = time.time() - start

# Single-item loop
start = time.time()
for t in sample_texts:
    pretrained.predict(t)
loop_time = time.time() - start

print(f"1,000 texts:")
print(f"  Batch predict:  {batch_time*1000:.1f} ms  ({1000/batch_time:,.0f} texts/sec)")
print(f"  Loop predict:   {loop_time*1000:.1f} ms  ({1000/loop_time:,.0f} texts/sec)")
print(f"  Speedup:        {loop_time/batch_time:.1f}x")
print(f"\nTip: Always use batch predict for throughput-sensitive workloads.")

---
## 8. Why Character N-grams Work for Language ID

Let's visualize the most discriminative character n-grams for a few languages.

In [ ]:
from collections import Counter

def extract_char_ngrams(text, n):
    text = text.lower().replace(' ', '_')  # treat spaces as word boundary
    return [text[i:i+n] for i in range(len(text)-n+1)]

# Sample 500 texts per language from training set
lang_texts = defaultdict(list)
for ex in train_ds:
    if len(lang_texts[ex['labels']]) < 500:
        lang_texts[ex['labels']].append(ex['text'])

# Find n-grams (n=3) that strongly distinguish each language
focus_langs = ['en', 'fr', 'de', 'fi', 'ar']

# Count 3-grams per language
lang_ngram_counts = {}
for lang in focus_langs:
    combined = ' '.join(lang_texts[lang])
    ngrams = extract_char_ngrams(combined, 3)
    lang_ngram_counts[lang] = Counter(ngrams)

# Compute total counts across languages to find distinctive ones
total_counts = Counter()
for counts in lang_ngram_counts.values():
    total_counts.update(counts)

# Distinctiveness: freq in lang / freq overall
fig, axes = plt.subplots(1, len(focus_langs), figsize=(16, 5))
for ax, lang in zip(axes, focus_langs):
    lang_counts_this = lang_ngram_counts[lang]
    total_this = sum(lang_counts_this.values())
    distinctive = [
        (ng, lang_counts_this[ng] / total_counts[ng])
        for ng in lang_counts_this
        if total_counts[ng] >= 50 and lang_counts_this[ng] >= 20
    ]
    top = sorted(distinctive, key=lambda x: -x[1])[:10]

    if top:
        ngs, scores = zip(*top)
        ax.barh(list(ngs), list(scores), color='#3498db', alpha=0.8)
        ax.set_xlim(0, 1)
        ax.set_title(f"{LANG_NAMES.get(lang, lang)} [{lang}]")
        ax.set_xlabel('Fraction in this lang')
    else:
        ax.set_title(f"{lang} (no data)")

plt.suptitle("Top Distinctive Character Trigrams per Language", fontsize=13)
plt.tight_layout()
plt.show()
print("Character n-grams capture scripts, diacritics, and spelling patterns — highly language-specific!")

---
## Summary

### What we covered

| Topic | Key takeaway |
|---|---|
| **FastText subwords** | Character n-grams capture script/diacritic signals — ideal for language ID |
| **Pre-trained model** | `lid.176.ftz` (~900 KB): 176 languages, high accuracy, ready to use |
| **Wikimedia sitematrix API** | Discover 1,000+ language editions; each has `code`, `name`, `localname` |
| **Wikipedia as training data** | Fetch article extracts per language → free labeled corpus for any language |
| **Training format** | `__label__lang text per line`, one example per line |
| **Key params** | `minn`/`maxn` for char n-grams, `wordNgrams`, `loss='ova'` for many classes |
| **Confidence threshold** | Filter unreliable predictions (short texts, code snippets) |
| **Code-switching** | Split into sentences and detect per-sentence |
| **Batch prediction** | Pass a list to `model.predict()` for large throughput gains |
| **Quantization** | `model.quantize()` shrinks the model 10–100× with minimal accuracy loss |

### When to use the pre-trained model vs train your own

| Use case | Recommendation |
|---|---|
| General-purpose, 176 languages | Pre-trained `lid.176` |
| Domain-specific text (code, tweets, medical) | Fine-tune on your domain |
| Rare / low-resource language | Collect Wikipedia articles via API and train |
| Dialect or script variants | Collect domain data and retrain |

### References

- Joulin et al. (2016) — [Bag of Tricks for Efficient Text Classification](https://arxiv.org/abs/1607.01759)
- Joulin et al. (2016) — [FastText.zip: Compressing text classification models](https://arxiv.org/abs/1612.03651)
- [FastText official documentation](https://fasttext.cc/docs/en/language-identification.html)
- [Wikimedia sitematrix API](https://commons.wikimedia.org/w/api.php?action=help&modules=sitematrix)
- Dataset: [`papluca/language-identification`](https://huggingface.co/datasets/papluca/language-identification) on Hugging Face